# Implement development lifecycle processes in Azure Databricks

## Apply Git version control best practices

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-development-lifecycle-processes-azure-databricks.apply-version-control-git.png)

### Demo: Configure Git Credentials

> **Trainer Note**: Live-demo in the Azure Databricks workspace UI.

1. Click your **avatar** (top-right) → **Settings** → **Linked accounts**
2. Under **Git provider**, select your provider (e.g., *GitHub*, *Azure DevOps*)
3. Enter your **username** and a **Personal Access Token (PAT)** with `repo` scope
4. Click **Save**

> **Tip**: Each developer links their own PAT — tokens must never be shared across accounts.


### Demo: Create a Git Folder (Clone a Repository)

> **Trainer Note**: Live-demo in the Azure Databricks workspace UI.

1. Sidebar → **Workspace** → navigate to your home folder
2. **Create** → **Git folder**
3. Enter the repository URL, e.g. `https://github.com/<org>/<repo>.git`
4. Select the Git provider and give the folder a name
5. Click **Create Git folder**

The repository contents appear immediately in the workspace.  
Look for the **branch indicator** next to the notebook title — this opens the Git dialog.


### Demo: Pull Changes & Keep the Repository Organized

> **Trainer Note**: Open an existing Git folder, launch the Git dialog, and pull changes.

**To pull the latest changes from the remote:**
1. Click the **branch indicator** next to any notebook title to open the Git dialog
2. Click **Pull** — Databricks fetches and merges remote changes automatically
3. Resolve any conflicts if prompted (the dialog lists all conflicting files)

**Sample `.gitignore` for a Databricks project:**

```
# Python artefacts
__pycache__/
*.py[cod]
*.egg-info/

# Databricks state
.databricks/
*.ipynb_checkpoints/

# Secrets — never commit credentials
.env
secrets/
```

> Stale remote branches are cleaned from local workspace after ~30 days.  
> Delete merged branches manually in your Git provider to keep the branch list manageable.


## Manage branching and pull requests

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-development-lifecycle-processes-azure-databricks.manage-branching-pull-requests-conflicts.png)

### Demo: Create a Feature Branch

> **Trainer Note**: Open any notebook inside a Git folder, then click the branch indicator.

**Recommended naming conventions:**

| Pattern | Example |
|---|---|
| `feature/<description>` | `feature/add-silver-layer` |
| `bugfix/<description>` | `bugfix/fix-null-customer-id` |
| `users/<name>/<description>` | `users/jane/experiment-aggregation` |

**Steps:**
1. Click the **branch button** (next to the notebook title) → **Create branch**
2. Enter a descriptive name, e.g. `feature/add-silver-layer`
3. Base it on `main` → click **Create**

> **Important**: Switching branches may delete workspace assets that don't exist on the target branch.  
> Always verify the branch has the assets you need before switching.


### Demo: Commit and Push Changes

> **Trainer Note**: Make a small edit in a notebook, then open the Git dialog to show the diff.

1. Edit a notebook cell (e.g., change a transformation value)
2. Click the **branch button** → the Git dialog opens, listing all modified files
3. In the **diff view**, confirm only the intended changes are shown
4. Enter a commit message following the convention:  
   `feat: Add order amount bucketing to silver layer`
5. Click **Commit & Push**

> **Note**: Notebook outputs are **not** committed when using source-file formats (`.py`, `.sql`).  
> Use `.ipynb` format only when you explicitly need to version output cells.


### Demo: Pull Request Workflow & Conflict Resolution

#### Pull Request Workflow (GitHub / Azure DevOps)

1. Push your feature branch to the remote repository
2. Open your Git provider → **New Pull Request**: `feature/add-silver-layer` → `main`
3. Assign reviewers, add a clear description of what changed and why
4. Address review feedback by pushing additional commits to the same branch
5. **Merge** after approval, then delete the branch

> Pull frequently from `main` to catch conflicts early — small merges are far easier to resolve.

#### Resolving a Merge Conflict in the Git Dialog

When a conflict is detected, conflicting files are listed in the Git dialog.

**Option A — Auto resolve**: Click the kebab menu (⋮) next to the file → **Keep all current changes** or **Take all incoming changes**

**Option B — Manual resolve**: Git marks the conflicting section:

```
<<<<<<< HEAD
result = df.groupBy("region").agg(F.sum("amount").alias("total"))
=======
result = df.groupBy("region").agg(F.avg("amount").alias("avg_amount"))
>>>>>>> feature/add-silver-layer
```

Edit the file to keep the correct logic, remove the conflict markers,  
then click **Mark as Resolved** → **Continue Merge**.


## Implement testing strategy

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-development-lifecycle-processes-azure-databricks.implement-testing-strategy.png)

### Demo: Unit Testing with pytest

We define a simple transformation function and write `pytest` unit tests against it.  
Tests use **synthetic DataFrames** — no production data required.


In [ ]:
%pip install pytest -q


In [ ]:
import os

# ── Transformation function ───────────────────────────────────────────────────
transforms_src = (
    'import pandas as pd\n\n'
    'def filter_by_region(df, region="North America"):\n'
    '    """Return rows where customer_region matches the given region."""\n'
    '    return df[df["customer_region"] == region]\n'
)

# ── pytest test file ──────────────────────────────────────────────────────────
test_src = (
    'import sys, os\n'
    'sys.path.insert(0, "/tmp/demo_transforms")\n'
    'import pandas as pd\n'
    'from transforms import filter_by_region\n\n'
    'def test_default_region_returns_correct_rows():\n'
    '    df = pd.DataFrame({\n'
    '        "customer_region": ["North America", "North America", "Europe", "Asia Pacific"],\n'
    '        "order_amount":     [100.0, 200.0, 150.0, 175.0]\n'
    '    })\n'
    '    result = filter_by_region(df)\n'
    '    assert len(result) == 2\n'
    '    assert all(result["customer_region"] == "North America")\n\n'
    'def test_specific_region_filter():\n'
    '    df = pd.DataFrame({\n'
    '        "customer_region": ["North America", "Europe", "Asia Pacific"],\n'
    '        "order_amount":     [100.0, 150.0, 175.0]\n'
    '    })\n'
    '    result = filter_by_region(df, region="Europe")\n'
    '    assert len(result) == 1\n'
    '    assert result.iloc[0]["order_amount"] == 150.0\n\n'
    'def test_unknown_region_returns_empty_dataframe():\n'
    '    df = pd.DataFrame({\n'
    '        "customer_region": ["North America", "Europe"],\n'
    '        "order_amount":     [100.0, 150.0]\n'
    '    })\n'
    '    result = filter_by_region(df, region="Antarctica")\n'
    '    assert len(result) == 0\n'
)

os.makedirs("/tmp/demo_transforms", exist_ok=True)
with open("/tmp/demo_transforms/transforms.py", "w") as f:
    f.write(transforms_src)
with open("/tmp/demo_transforms/test_transforms.py", "w") as f:
    f.write(test_src)

print("Success: /tmp/demo_transforms/transforms.py written")
print("Success: /tmp/demo_transforms/test_transforms.py written")


In [ ]:
import pytest

# Run the unit test suite
retcode = pytest.main(["/tmp/demo_transforms/test_transforms.py", "-v", "-p", "no:cacheprovider"])
assert retcode == 0, "Some unit tests failed — review the output above."


### Demo: Integration Test

An integration test reads from the **orders** table (created by `setup_01(spark)`) and validates its schema and row count.  
This confirms that the ingestion step wrote data in the expected format.

> **Trainer Note**: Run `setup_01(spark)` from the setup script before executing the cell below.


In [ ]:
# ── Integration test: validate orders table schema and row count ─────────────

def test_orders_table_integration():
    df = spark.sql("SELECT * FROM trainer_demo.demo_01.orders LIMIT 10")

    expected_columns = ["order_id", "customer_region", "order_date", "order_amount"]
    assert list(df.columns) == expected_columns, \
        f"Schema mismatch — expected {expected_columns}, got {list(df.columns)}"

    total_rows = spark.sql(
        "SELECT COUNT(*) AS cnt FROM trainer_demo.demo_01.orders"
    ).first().cnt
    assert total_rows > 0, "orders table is empty — run setup_01(spark) first"

    print(f"  Schema : {list(df.columns)}")
    print(f"  Rows   : {total_rows:,}")
    print("Integration test passed")

test_orders_table_integration()


### Demo: User Acceptance Testing (UAT)

A UAT scenario lets **stakeholders validate** that the pipeline output satisfies business requirements.  
Here we verify that total order revenue falls within a realistic range — simulating a finance reconciliation check.


In [ ]:
# ── UAT Scenario: revenue sanity check ──────────────────────────────────────
result = spark.sql("""
    SELECT
        COUNT(*)                        AS total_orders,
        ROUND(SUM(order_amount), 2)     AS total_revenue,
        ROUND(MIN(order_amount), 2)     AS min_order,
        ROUND(MAX(order_amount), 2)     AS max_order,
        COUNT(DISTINCT customer_region)  AS distinct_regions
    FROM trainer_demo.demo_01.orders
""").first()

print(f"Total Orders    : {result.total_orders:,}")
print(f"Total Revenue   : ${result.total_revenue:,.2f}")
print(f"Order Range     : ${result.min_order:.2f} - ${result.max_order:.2f}")
print(f"Distinct Regions: {result.distinct_regions}")

# ── Acceptance criteria ───────────────────────────────────────────────────────
expected_min = 10_000.0
expected_max = 10_000_000.0

if expected_min <= result.total_revenue <= expected_max:
    print("\nUAT PASSED: Revenue total is within the accepted business range")
else:
    print(f"\nUAT FAILED: Revenue ${result.total_revenue:,.2f} is outside "
          f"[${expected_min:,.0f} - ${expected_max:,.0f}]")


## Configure and package DABs

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-development-lifecycle-processes-azure-databricks.configure-package-databricks-asset-bundles.png)

### Demo: Bundle Directory Structure

A Databricks Asset Bundle (DAB) lives alongside your source code in a Git repository.

```
orders-pipeline/           <- repository root
├── databricks.yml         <- bundle identity, workspace host(s), targets, includes
├── resources/
│   └── jobs.yml           <- job & pipeline resource definitions
├── notebooks/
│   ├── ingest.py
│   └── transform.py
└── requirements.txt
```

> **Trainer Note**: Show the VS Code **Databricks extension** → *Bundle Explorer* panel  
> to navigate the bundle structure and validate configuration without using the CLI.


In [ ]:
# ── Example: databricks.yml — root bundle configuration ──────────────────────
print("""
# databricks.yml
bundle:
  name: orders-pipeline

include:
  - resources/*.yml

workspace:
  host: https://adb-<dev-workspace-id>.azuredatabricks.net

targets:
  dev:
    default: true          # used when no -t flag is provided

  prod:
    workspace:
      host: https://adb-<prod-workspace-id>.azuredatabricks.net
""")


In [ ]:
# ── Example: resources/jobs.yml — variables + task dependencies ─────────────
print("""
# resources/jobs.yml
variables:
  environment:
    description: Deployment environment name
    default: dev
  cluster_id:
    description: Cluster ID for task execution
    default: 1234-567890-abcde123

resources:
  jobs:
    orders-etl:
      name: ${var.environment}-orders-etl
      tasks:
        - task_key: ingest
          existing_cluster_id: ${var.cluster_id}
          notebook_task:
            notebook_path: ./notebooks/ingest.py

        - task_key: transform
          depends_on:
            - task_key: ingest     # task dependency
          existing_cluster_id: ${var.cluster_id}
          notebook_task:
            notebook_path: ./notebooks/transform.py
            base_parameters:
              env: ${var.environment}

targets:
  dev:
    variables:
      cluster_id: 1234-567890-abcde123
      environment: dev
  prod:
    variables:
      cluster_id: 9876-543210-zyxwv987
      environment: prod
""")


## Deploy bundle with Databricks CLI

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-development-lifecycle-processes-azure-databricks.deploy-bundle-databricks-cli.png)

### Demo: Databricks CLI — Validate, Plan, Deploy

> **Trainer Note**: Run the commands below in a **local terminal** where the Databricks CLI (v0.200+) is installed  
> and you have authenticated with `databricks auth login`.  
> In a CI/CD pipeline these same commands run unattended using service-principal credentials.

**Install the CLI** (if not already installed):

```bash
# macOS (Homebrew)
brew tap databricks/tap
brew install databricks

# Verify installation
databricks --version
```


#### Step 1 — Validate bundle configuration

```bash
# Check YAML syntax and flag missing required fields
databricks bundle validate
```

Expected output:
```
Name: orders-pipeline
Target: dev
Workspace:
  Host: https://adb-<id>.azuredatabricks.net
  User: you@example.com
  Path: /Users/you@example.com/.bundle/orders-pipeline/dev

Validation OK!
```

> Fix any errors or warnings before proceeding to deployment.


#### Step 2 — Preview deployment changes

```bash
# Show what will be created, updated, or deleted — no changes are made
databricks bundle plan

# Preview for a specific target
databricks bundle plan -t prod
```

Example output:
```
Building python_artifact...
create jobs.orders-etl
```

> If you see unexpected **deletes**, review your `databricks.yml` before proceeding.


#### Step 3 — Deploy to a target workspace

```bash
# Deploy to the default target (dev)
databricks bundle deploy

# Deploy to a named target
databricks bundle deploy -t prod

# Skip confirmation prompts — suitable for automated CI/CD pipelines
databricks bundle deploy -t prod --auto-approve
```

The CLI will:
- **Create** new resources defined in the configuration
- **Update** existing resources that were previously deployed
- **Delete** resources that no longer appear in the configuration

> Each deployment has a unique identity based on bundle name, target, and deploying user.  
> Coordinate with your team to avoid conflicting deployments to shared environments.


#### Step 4 — Verify deployed resources

```bash
# List all deployed resources with direct links to the Databricks UI
databricks bundle summary

# Open a specific resource directly in your browser
databricks bundle open orders-etl
```

Example `bundle summary` output:
```
Resources:
  Jobs:
    orders-etl:
      Name: [dev you] orders-etl
      URL:  https://adb-<id>.azuredatabricks.net/jobs/123456789
```


#### Step 5 — Troubleshoot common deployment issues

```bash
# Re-authenticate if you see "permission denied" or token-expired errors
databricks auth login --host https://adb-<id>.azuredatabricks.net

# Override a stuck deployment lock (left by an interrupted deploy)
databricks bundle deploy --force-lock

# Pass a variable value at deploy time without editing databricks.yml
databricks bundle deploy -t prod --var="cluster_id=9876-543210-zyxwv987"
```

| Error | Cause | Fix |
|---|---|---|
| Permission denied | Expired / missing PAT | `databricks auth login` |
| Lock conflict | Previous deploy interrupted | `--force-lock` |
| Active run conflict | Job running during deploy | Wait, or use `--fail-on-active-runs` |
| Unknown property warning | CLI version mismatch | `brew upgrade databricks` |
